# pic0rick RP2350 DSP firmware — step-by-step example

Notebook version of `example_dsp.py`. Targets the **`-DDSP`** firmware build
(see `docs/dsp_test_guide.md`). Run the cells top to bottom.

Uses the new API: `Pic0rick.status()`, `.read_raw()` (8000 raw ADC),
`.read_fft()` (4096-sample Hilbert envelope), and the pulser.

Requires: `pyserial`, `numpy`, `matplotlib`.

## 1. Imports

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from pic0rick.device import Pic0rick

## 2. Connect

Leave `PORT = None` to auto-detect the USB-CDC port, or set it explicitly
(e.g. `"/dev/ttyACM0"`, `"COM7"`).

In [ ]:
PORT = None
probe = Pic0rick(port=PORT, verbose=False)

## 3. Status

`status()` returns a parsed dict. It only works on a **DSP** build — a
`ValueError` here means the board is running the stdio (non-DSP) firmware; flash
a `rp2350-*-dsp` UF2 (see the test guide).

In [ ]:
st = probe.status()
print(st['board'], st['package'], '| firmware', st['firmware'],
      '| backend', st['dsp_backend'])
st

## 4. Set the TGC gain

Raw 10-bit DAC value 0..1023 (spi1 MCP4812). The DSP build's command is
`dac write <n>` (the stdio build uses `write dac`, i.e. `Pic0rick.dac()`).

In [ ]:
probe.ser.write(b'dac write 300\n')
probe.sread()

## 5. `read_raw` — 8000 raw ADC samples (no FFT)

`.samples()` returns a numpy `uint16` array; the frame's CRC was already
checked by the reader.

In [ ]:
frame = probe.read_raw()
raw = frame.samples()
print('samples', frame.header.sample_count, '| min/mean/max',
      int(raw.min()), int(raw.mean()), int(raw.max()))
plt.figure(figsize=(9, 3))
plt.plot(raw)
plt.title('read_raw (8000 ADC samples)')
plt.xlabel('sample'); plt.ylabel('ADC code (10-bit)')
plt.show()

## 6. `read_fft` — 4096-sample Hilbert envelope

`.samples()` returns a numpy `float32` envelope.

In [ ]:
frame = probe.read_fft()
env = frame.samples()
print('samples', frame.header.sample_count, '| peak',
      round(float(env.max()), 1), '| dc_mean', round(frame.header.adc_dc_mean, 1))
plt.figure(figsize=(9, 3))
plt.plot(env)
plt.title('read_fft (4096-pt Hilbert envelope)')
plt.xlabel('sample'); plt.ylabel('envelope (ADC counts)')
plt.show()

## 7. Pulsed acquisition

Configure and arm the pulser, then capture an envelope. Compare it with the
idle envelope from the previous cell.

In [ ]:
probe.ser.write(b'pulse config 96 6000 96 neg-first\n'); probe.sread()
probe.ser.write(b'pulser arm\n'); probe.sread()
env_pulsed = probe.read_fft().samples()
probe.ser.write(b'pulser disarm\n'); probe.sread()
print('idle peak', round(float(env.max()), 1),
      '| pulsed peak', round(float(env_pulsed.max()), 1))
plt.figure(figsize=(9, 3))
plt.plot(env, label='idle')
plt.plot(env_pulsed, label='pulser armed')
plt.title('read_fft envelope: idle vs pulser armed')
plt.xlabel('sample'); plt.ylabel('envelope'); plt.legend()
plt.show()

## 8. (Optional) save the captures

In [ ]:
np.save('raw.npy', raw)
np.save('envelope.npy', env)
print('saved raw.npy and envelope.npy')

---
See `docs/dsp_test_guide.md` for the full command set, `pic0rick.dsp` for the
frame/protocol details, and `python/example_dsp.py` for the same flow as a
runnable script.